In [6]:
import pandas as pd
import numpy as np
import scipy.stats as stats

In [7]:
# 🔹 Calculate Mean Expression
def CalculateMeanExpression(expr_data):
    mean_expr = pd.DataFrame(expr_data.mean(axis=1), columns=['mean_expr']).reset_index()
    return mean_expr

# 🔹 Calculate Expression Variance
def CalculateVarianceExpression(expr_data):
    var_expr = pd.DataFrame(expr_data.var(axis=1), columns=['expr_variance']).reset_index()
    return var_expr

# 🔹 Expand gene pairs with mean and variance expression
def ExpandToPairs(mean_expr, var_expr, pairs):
    # Add mean expression
    pairs_expr = pd.merge(pairs, mean_expr.rename(columns={'ensembl_id':'A1_ensembl', 'mean_expr':'A1_mean_expr'}), how='left')
    pairs_expr = pd.merge(pairs_expr, mean_expr.rename(columns={'ensembl_id':'A2_ensembl', 'mean_expr':'A2_mean_expr'}), how='left')

    # Add variance expression
    pairs_expr = pd.merge(pairs_expr, var_expr.rename(columns={'ensembl_id':'A1_ensembl', 'expr_variance':'A1_expr_variance'}), how='left')
    pairs_expr = pd.merge(pairs_expr, var_expr.rename(columns={'ensembl_id':'A2_ensembl', 'expr_variance':'A2_expr_variance'}), how='left')

    print('Pairs where both genes have expr:', 
          pairs_expr[(~pairs_expr.A1_mean_expr.isna()) & (~pairs_expr.A2_mean_expr.isna())].shape[0], '/', pairs_expr.shape[0])
    
    # Filter pairs down to those where both have non-zero, non-NA expression
    pairs_expr_subset = pairs_expr[
        (~pairs_expr.A1_mean_expr.isna()) & (~pairs_expr.A2_mean_expr.isna()) & 
        (pairs_expr.A1_mean_expr != 0) & (pairs_expr.A2_mean_expr != 0)
    ]
    print('# Pairs where both have expression:', pairs_expr_subset.shape[0])
    print('# Total Pairs:', pairs.shape[0])

    return pairs_expr

# 🔹 Original Spearman Correlation Function
def CalculateExprCorrelation(pairs, expr_data):
    mean_expr = CalculateMeanExpression(expr_data)
    var_expr = CalculateVarianceExpression(expr_data)

    pairs_expr_subset = ExpandToPairs(mean_expr, var_expr, pairs)

    def compute_spearman_corr(pair):
        return stats.spearmanr(expr_data.loc[pair.A1_ensembl], expr_data.loc[pair.A2_ensembl])[0]

    features_spearman = pairs_expr_subset.assign(spearman_corr = pairs_expr_subset.apply(compute_spearman_corr, axis=1))
    return features_spearman

# 🔹 Fast Spearman Correlation Version
def CalculateExprCorrelation_Fast(pairs, expr_data):
    mean_expr = CalculateMeanExpression(expr_data)
    var_expr = CalculateVarianceExpression(expr_data)

    pairs_expr_subset = ExpandToPairs(mean_expr, var_expr, pairs)

    expr_ranks = expr_data.rank(axis=1)  # Rank across samples (columns)
    expr_dict = expr_ranks.to_dict(orient="index")

    def fast_spearman(pair):
        A1 = expr_dict.get(pair.A1_ensembl, None)
        A2 = expr_dict.get(pair.A2_ensembl, None)
        if A1 is None or A2 is None:
            return np.nan
        return np.corrcoef(list(A1.values()), list(A2.values()))[0, 1]

    pairs_expr_subset["spearman_corr"] = pairs_expr_subset.apply(fast_spearman, axis=1)
    return pairs_expr_subset


# Expression Correlation


## Data Source: 
https://www.gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression (TPM) 

# BIOMARKER VS QUERY

In [ ]:
main_file_csv = pd.read_csv('datasets/PredictingSLResistanceFeatures_ATM_ATR_main.csv')
biomarker_query = main_file_csv[['ensembl_gene_id_biomarker','ensembl_gene_id_query']]
biomarker_query = biomarker_query.rename(columns={
    'ensembl_gene_id_biomarker': 'A1_ensembl',
    'ensembl_gene_id_query' : 'A2_ensembl'
})
biomarker_query_genes = pd.unique(biomarker_query[['A1_ensembl', 'A2_ensembl']].values.ravel())

df_biomarker_query_genes = pd.DataFrame()

for chunk in pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz', sep='\t', skiprows=2, iterator=True, chunksize=1000):
    chunk_df = chunk.assign(ensembl_id = chunk.Name.apply(lambda x: x.split('.')[0])).drop(columns=['Description'])
    df_biomarker_query_genes = pd.concat([df_biomarker_query_genes, chunk_df[chunk_df.ensembl_id.isin(biomarker_query_genes)].set_index('ensembl_id')])
duplicate_biomarker_query = df_biomarker_query_genes.index[df_biomarker_query_genes.index.duplicated()]
df_biomarker_query_genes = df_biomarker_query_genes.reset_index()
duplicates = df_biomarker_query_genes[df_biomarker_query_genes.duplicated(subset=["ensembl_id"], keep=False)]
non_zero_rows = duplicates[(duplicates.iloc[:, 2:] != 0).any(axis=1)]
expr_data_biomarker_query = df_biomarker_query_genes.drop(non_zero_rows.index)

expr_data_biomarker_query = expr_data_biomarker_query.set_index("ensembl_id")
expr_data_biomarker_query.drop(columns=['Name'], inplace = True)
expr_data_biomarker_query

biomarker_query_expression = CalculateExprCorrelation_Fast(biomarker_query,expr_data_biomarker_query)
biomarker_query_expression.to_csv('feature_output/gtex_co_expression_biomarker_query_ATM_ATR.csv',index=False)

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_83175/2631599570.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')


array(['ENSG00000171862', 'ENSG00000105647', 'ENSG00000165458', ...,
       'ENSG00000206077', 'ENSG00000214189', 'ENSG00000204532'],
      dtype=object)

In [8]:
def run_biomarker_query_expression_from_gtex(main_csv_path: str, expr_data_path: str, output_prefix: str):
    """
    Compute GTEx-based biomarker–query co-expression features for a given SL pair.

    Parameters
    ----------
    main_csv_path : str
        Path to the SL dataset (e.g. datasets/PredictingSLResistanceFeatures_ATM_ATR_main.csv).
    expr_data_path : str
        Path to GTEx expression file (gzipped GCT format).
    output_prefix : str
        Prefix for output file (e.g. "ATM_ATR").

    Returns
    -------
    biomarker_query_expression : pd.DataFrame
    """
    # Load dataset
    main_file_csv = pd.read_csv(main_csv_path, low_memory=False)

    # Extract biomarker–query pairs
    biomarker_query = main_file_csv[['ensembl_gene_id_biomarker', 'ensembl_gene_id_query']].rename(
        columns={'ensembl_gene_id_biomarker': 'A1_ensembl', 'ensembl_gene_id_query': 'A2_ensembl'}
    )
    biomarker_query_genes = pd.unique(biomarker_query[['A1_ensembl', 'A2_ensembl']].values.ravel())

    # Collect expression data for these genes
    df_biomarker_query_genes = pd.DataFrame()
    for chunk in pd.read_csv(expr_data_path, sep='\t', skiprows=2, iterator=True, chunksize=1000):
        chunk_df = chunk.assign(ensembl_id=chunk.Name.apply(lambda x: x.split('.')[0])).drop(columns=['Description'])
        df_biomarker_query_genes = pd.concat([
            df_biomarker_query_genes,
            chunk_df[chunk_df.ensembl_id.isin(biomarker_query_genes)].set_index('ensembl_id')
        ])

    # Handle duplicates
    df_biomarker_query_genes = df_biomarker_query_genes.reset_index()
    duplicates = df_biomarker_query_genes[df_biomarker_query_genes.duplicated(subset=["ensembl_id"], keep=False)]
    non_zero_rows = duplicates[(duplicates.iloc[:, 2:] != 0).any(axis=1)]
    expr_data_biomarker_query = df_biomarker_query_genes.drop(non_zero_rows.index)

    # Format expression dataframe
    expr_data_biomarker_query = expr_data_biomarker_query.set_index("ensembl_id")
    expr_data_biomarker_query.drop(columns=['Name'], inplace=True)

    # Calculate expression correlation
    biomarker_query_expression = CalculateExprCorrelation_Fast(biomarker_query, expr_data_biomarker_query)

    # Save output
    biomarker_query_expression.to_csv(
        f'feature_output/gtex_co_expression_biomarker_query_{output_prefix}.csv',
        index=False
    )

    print(f"✅ Expression features saved: gtex_co_expression_biomarker_query_{output_prefix}.csv")
    return biomarker_query_expression


In [9]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel with Biomarker + Target columns
expr_data_path = "/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz"

pairs_df = pd.read_excel(excel_file)

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_biomarker_query_expression_from_gtex(
        main_csv_path=main_file,
        expr_data_path=expr_data_path,
        output_prefix=SL_pair
    )


Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features saved: gtex_co_expression_biomarker_query_MSH6_WRN.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features saved: gtex_co_expression_biomarker_query_CDH1_ROS1.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


...

✅ Expression features saved: gtex_co_expression_biomarker_query_BRCA1_USP1.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features saved: gtex_co_expression_biomarker_query_BRCA1_POLQ.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features saved: gtex_co_expression_biomarker_query_KRAS_PLK1.csv


In [10]:
def run_target1_query_expression_from_gtex(main_csv_path: str, expr_data_path: str, output_prefix: str):
    """
    Compute GTEx-based co-expression features for target1–query pairs.

    Parameters
    ----------
    main_csv_path : str
        Path to the SL dataset (e.g. datasets/PredictingSLResistanceFeatures_ATM_ATR_main.csv).
    expr_data_path : str
        Path to GTEx expression file (gzipped GCT format).
    output_prefix : str
        Prefix for output file (e.g. "ATM_ATR").

    Returns
    -------
    target1_query_expression : pd.DataFrame
    """
    # Load dataset
    main_file_csv = pd.read_csv(main_csv_path, low_memory=False)

    # --- Target1–Query pairs ---
    target1_query = main_file_csv[['ensembl_gene_id_target1', 'ensembl_gene_id_query']].rename(
        columns={'ensembl_gene_id_target1': 'A1_ensembl', 'ensembl_gene_id_query': 'A2_ensembl'}
    )
    target1_query_genes = pd.unique(target1_query[['A1_ensembl', 'A2_ensembl']].values.ravel())

    # Collect expression data for these genes
    df_target1_query_genes = pd.DataFrame()
    for chunk in pd.read_csv(expr_data_path, sep='\t', skiprows=2, iterator=True, chunksize=1000):
        chunk_df = chunk.assign(ensembl_id=chunk.Name.apply(lambda x: x.split('.')[0])).drop(columns=['Description'])
        df_target1_query_genes = pd.concat([
            df_target1_query_genes,
            chunk_df[chunk_df.ensembl_id.isin(target1_query_genes)].set_index('ensembl_id')
        ])

    # Handle duplicates
    df_target1_query_genes = df_target1_query_genes.reset_index()
    duplicates = df_target1_query_genes[df_target1_query_genes.duplicated(subset=["ensembl_id"], keep=False)]
    non_zero_rows = duplicates[(duplicates.iloc[:, 2:] != 0).any(axis=1)]
    expr_data_target1_query = df_target1_query_genes.drop(non_zero_rows.index)

    # Format expression dataframe
    expr_data_target1_query = expr_data_target1_query.set_index("ensembl_id")
    expr_data_target1_query.drop(columns=['Name'], inplace=True)

    # Calculate expression correlation
    target1_query_expression = CalculateExprCorrelation_Fast(target1_query, expr_data_target1_query)

    # Save output
    target1_query_expression.to_csv(
        f'feature_output/gtex_co_expression_target1_query_{output_prefix}.csv',
        index=False
    )

    print(f"✅ Expression features (target1–query) saved: gtex_co_expression_target1_query_{output_prefix}.csv")
    return target1_query_expression


In [11]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel with Biomarker + Target columns
expr_data_path = "/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz"

pairs_df = pd.read_excel(excel_file)

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_target1_query_expression_from_gtex(
        main_csv_path=main_file,
        expr_data_path=expr_data_path,
        output_prefix=SL_pair
    )


Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features (target1–query) saved: gtex_co_expression_target1_query_MSH6_WRN.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features (target1–query) saved: gtex_co_expression_target1_query_CDH1_ROS1.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


...

✅ Expression features (target1–query) saved: gtex_co_expression_target1_query_BRCA1_USP1.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features (target1–query) saved: gtex_co_expression_target1_query_BRCA1_POLQ.csv
Pairs where both genes have expr: 19192 / 19271
# Pairs where both have expression: 19141
# Total Pairs: 19271


/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/metinyazar/anaconda3/envs/main/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✅ Expression features (target1–query) saved: gtex_co_expression_target1_query_KRAS_PLK1.csv
